In [1]:
import os

# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics
import subprocess
import sys
import seaborn as sns
import numpy as np
from scipy.sparse import csr_matrix
import scanpy.external as sce
from sklearn.metrics import silhouette_score
import datetime
import numpy as np
from collections import defaultdict
import scipy.sparse as sp
from collections import defaultdict
import gffutils 
import scipy.sparse as sp
import gffutils 

# sys.path.append('../utils')
# from functions import * 

%config InlineBackend.print_figure_kwargs={'facecolor' : "w"}
%config InlineBackend.figure_format='retina'

# Flag for whether to also read intron and exons files 
read_introns_exons = True

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis/Human_Splicing_Foundation/GeneExpression


In [2]:
# Load gene gtf file for gene length 
gtf_hg38 = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/human-reference/gencode/gencode.v45.primary_assembly.annotation.gtf"
db_file = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/human-reference/gencode/gencode_hg38.db"

WD="/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression"
today = datetime.datetime.now()
today = today.strftime("%Y-%m-%d")

In [3]:
def extract_gene_transcript_info(gtf_file, db_file):
    """
    Parses a GENCODE GTF file to compute:
    - Mean transcript length per gene (sum of exons)
    - Mean intron length per gene (transcript span - exon length)
    - Number of transcripts per gene
    - Transcript biotypes
    - Gene name
    
    Returns a DataFrame with gene_id, gene_name, mean_transcript_length, mean_intron_length, 
    num_transcripts, and transcript_biotypes.
    """

    if os.path.exists(db_file):
        print("Using existing GTF database.")
        db = gffutils.FeatureDB(db_file, keep_order=True)
        print("Database loaded successfully!")
    else:
        print("Creating GTF database (this may take a few minutes)...")
        db = gffutils.create_db(
            gtf_file,
            db_file,
            force=True,
            keep_order=True,
            disable_infer_transcripts=False,
            disable_infer_genes=True
        )
        print("Database created successfully!")

    gene_exon_lengths = defaultdict(list)
    gene_intron_lengths = defaultdict(list)
    gene_names = {}
    gene_biotypes = defaultdict(set)
    transcript_counts = defaultdict(int)

    print("Processing transcripts to compute exon and intron lengths...")
    for transcript in tqdm(db.features_of_type("transcript"), desc="Processing Transcripts", unit=" transcript"):
        gene_id = transcript.attributes["gene_id"][0]
        gene_name = transcript.attributes.get("gene_name", ["unknown"])[0]
        transcript_biotype = transcript.attributes.get("transcript_type", ["unknown"])[0]

        exons = list(db.children(transcript, featuretype="exon", order_by="start"))
        if len(exons) == 0:
            continue  # skip transcripts with no exons

        exon_length = sum(exon.end - exon.start + 1 for exon in exons)
        transcript_start = exons[0].start
        transcript_end = exons[-1].end
        transcript_span = transcript_end - transcript_start + 1
        intron_length = transcript_span - exon_length  # includes gaps between exons

        gene_exon_lengths[gene_id].append(exon_length)
        gene_intron_lengths[gene_id].append(max(0, intron_length))  # avoid negative values
        gene_names[gene_id] = gene_name
        gene_biotypes[gene_id].add(transcript_biotype)
        transcript_counts[gene_id] += 1

    print("Finished processing transcripts.")

    gene_ids = list(gene_exon_lengths.keys())
    gene_info_df = pd.DataFrame({
        "gene_id": gene_ids,
        "gene_name": [gene_names[g] for g in gene_ids],
        "mean_transcript_length": [sum(gene_exon_lengths[g]) / len(gene_exon_lengths[g]) for g in gene_ids],
        "mean_intron_length": [sum(gene_intron_lengths[g]) / len(gene_intron_lengths[g]) for g in gene_ids],
        "num_transcripts": [transcript_counts[g] for g in gene_ids],
        "transcript_biotypes": [", ".join(sorted(gene_biotypes[g])) for g in gene_ids]
    })

    return gene_info_df

def preprocess_ab_adata(adata, metadata, dataset_label="allen_brain", 
                        metadata_key="sample_name", rename_var=True):
    """
    Standardizes Allen Brain adata object:
    - Subsets metadata to only matching cells
    - Renames gene info
    - Stores raw counts layer
    """
    adata = adata.copy()
    adata.var['gene_name'] = adata.var_names
    adata.obs["dataset"] = dataset_label

    # Use correct column to index metadata
    if metadata_key not in metadata.columns:
        raise ValueError(f"Metadata key '{metadata_key}' not found in metadata columns.")
    
    metadata_sub = metadata[metadata[metadata_key].isin(adata.obs_names)].copy()
    metadata_sub = metadata_sub.set_index(metadata_key)

    # Align metadata to adata
    adata = adata[adata.obs_names.isin(metadata_sub.index)].copy()
    adata.obs = metadata_sub.loc[adata.obs_names]

    # Rename gene_name -> gene_symbol
    if rename_var:
        adata.var.rename(columns={"gene_name": "gene_symbol"}, inplace=True)
    else:
        adata.var["gene_symbol"] = adata.var["gene_name"]
    adata.layers["raw_counts"] = adata.X.copy()
    print(f"Number of cells/nuclei: {adata.shape[0]}")
    return adata

### First, get gene info (average transcript length and average total intron length)

In [4]:
# Add gene length information to the adata object [done]
# follow recommended practices for smart-seq2 raw count normalization
# https://docs.scvi-tools.org/en/1.0.0/tutorials/notebooks/tabula_muris.html

# Load gene info
gene_info_df = extract_gene_transcript_info(gtf_hg38, db_file)
gene_info_df = gene_info_df.drop_duplicates(subset="gene_id")
gene_info_df = gene_info_df.drop_duplicates(subset="gene_name")

Using existing GTF database.
Database loaded successfully!
Processing transcripts to compute exon and intron lengths...


Processing Transcripts: 252989 transcript [04:26, 948.05 transcript/s] 


Finished processing transcripts.


### Prep the three anndata objects (total counts, exons only and introns only)

In [5]:
# Set up paths for files 
introns_only = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/intron.csv"
exons_only = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/exon.csv"
read_introns_exons = True 

if read_introns_exons:
    ab_adata_introns = sc.read_csv(introns_only)
    ab_adata_introns = ab_adata_introns.transpose()
    print(f"Done reading {introns_only}")

    ab_adata_exons = sc.read_csv(exons_only)
    ab_adata_exons = ab_adata_exons.transpose()
    print(f"Done reading {exons_only}")
    
# Read in metadata
metadata = "/gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/INFO/metadata.csv"
metadata = pd.read_csv(metadata)

# Process exon and intron counts (uses 'exp_component_name')
if read_introns_exons:
    ab_adata_introns = preprocess_ab_adata(ab_adata_introns, metadata, metadata_key="exp_component_name")
    ab_adata_exons = preprocess_ab_adata(ab_adata_exons, metadata, metadata_key="exp_component_name")

print("All AB datasets preprocessed with correct metadata mapping.")

Done reading /gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/intron.csv
Done reading /gpfs/commons/datasets/controlled/BRAIN_NeMO/lein-human-cortex/GeneExpression/exon.csv
Number of cells/nuclei: 49417
Number of cells/nuclei: 49417
All AB datasets preprocessed with correct metadata mapping.


### Read in the Tabula Sapien V2 dataset (SS2 cells only!)

In [6]:
# load in tabula sapien data 
# tabsap_adata = sc.read_h5ad("/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TS_figshare/TabulaSapiens.h5ad") # original tabula sapiens 
print(f"Now reading in Tabula Sapiens V2 data...")
tabsap_adata = sc.read_h5ad("/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/merged_tabula_sapiens.h5ad")
tabsap_adata.obs["dataset"] = "tabula_sapiens"

print("Number of duplicated gene symbols in tabsap_adata:", tabsap_adata.var["gene_symbol"].duplicated().sum())
# Find the rows in tabsap_adata.var that are duplicated
duplicated_genes = tabsap_adata.var[tabsap_adata.var["gene_symbol"].duplicated(keep=False)]
# Remove the duplicated genes from tabsap_adata.var
tabsap_adata = tabsap_adata[:, ~tabsap_adata.var.index.isin(duplicated_genes.index)].copy()
print("Number of duplicated gene symbols in tabsap_adata:", tabsap_adata.var["gene_symbol"].duplicated().sum())

tabsap_adata.var["gene_name"] = tabsap_adata.var["gene_symbol"]
tabsap_adata = tabsap_adata[:, tabsap_adata.var["gene_name"].isin(gene_info_df["gene_name"])].copy()
tabsap_adata.layers["raw_counts"] = tabsap_adata.X.copy()

print("Number of genes in common between the two datasets:", len(set(tabsap_adata.var["gene_symbol"]).intersection(set(ab_adata_exons.var["gene_symbol"]))))

Now reading in Tabula Sapiens V2 data...
Number of duplicated gene symbols in tabsap_adata: 1200
Number of duplicated gene symbols in tabsap_adata: 0
Number of genes in common between the two datasets: 30340


### Save all these anndatas so don't need to remake/clean up next time

In [7]:
# convert .X to sparse array in ab_adata, ab_adata_exons, ab_adata_introns
ab_adata_exons.X = csr_matrix(ab_adata_exons.X)
ab_adata_introns.X = csr_matrix(ab_adata_introns.X)

In [8]:
output_dir="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data" 
# make a dir called processed_data
os.makedirs(output_dir, exist_ok=True)

# write compressed file tabsap_adata to file 
tabsap_adata.write_h5ad(os.path.join(output_dir, f"tabsap_adata_{today}.h5ad"), compression="lzf")
print(f"Done saving tabsap_adata to {output_dir}")

ab_adata_exons.write_h5ad(os.path.join(output_dir, f"ab_adata_exons_{today}.h5ad"), compression="lzf")
print(f"Done saving ab_adata_exons to {output_dir}")

ab_adata_introns.write_h5ad(os.path.join(output_dir, f"ab_adata_introns_{today}.h5ad"), compression="lzf")
print(f"Done saving ab_adata_introns to {output_dir}")

print(f"Saved all anndata objects to {output_dir}")
print(f"Done processing all datasets.")

Done saving tabsap_adata to /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data
Done saving ab_adata_exons to /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data
Done saving ab_adata_introns to /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data
Saved all anndata objects to /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data
Done processing all datasets.


In [19]:
ab_adata_exons.var

,gene_symbol
3.8-1.2,3.8-1.2
3.8-1.3,3.8-1.3
3.8-1.4,3.8-1.4
3.8-1.5,3.8-1.5
5-HT3C2,5-HT3C2
...,...
ZYX,ZYX
ZZEF1,ZZEF1
ZZZ3,ZZZ3
bA255A11.4,bA255A11.4


In [16]:
# save gene_info_df to file 
gene_info_df.to_csv(os.path.join(output_dir, f"gene_info_df_{today}.csv"), index=False)
print(f"Done saving gene_info_df to {output_dir}")


Done saving gene_info_df to /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data


In [17]:
# read in gene_info_df
gene_info_df = pd.read_csv(os.path.join(output_dir, f"gene_info_df_{today}.csv"))
gene_info_df.head()


,gene_id,gene_name,mean_transcript_length,mean_intron_length,num_transcripts,transcript_biotypes
0,ENSG00000290825.1,DDX11L2,1657.0,884.0,1,lncRNA
1,ENSG00000223972.6,DDX11L1,632.0,1029.0,1,transcribed_unprocessed_pseudogene
2,ENSG00000227232.6,WASH7P,1380.0,8811.0,1,unprocessed_pseudogene
3,ENSG00000278267.1,MIR6859-1,68.0,0.0,1,miRNA
4,ENSG00000243485.5,MIR1302-2HG,623.5,570.0,2,lncRNA
